In [ ]:
%pip install mcp anthropic --quiet

## 1. Servidor MCP Simple

El siguiente código crea un servidor MCP con herramientas matemáticas.
Guárdalo en `mcp_math_server.py` y ejecútalo por separado.

In [ ]:
# Escribir el servidor MCP en un archivo
mcp_server_code = '''
# mcp_math_server.py
# Servidor MCP con herramientas matemáticas

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types
import math
import json

app = Server("math-server")

@app.list_tools()
async def list_tools() -> list[types.Tool]:
    """Lista todas las herramientas disponibles."""
    return [
        types.Tool(
            name="calcular",
            description="Realiza cálculos matemáticos",
            inputSchema={
                "type": "object",
                "properties": {
                    "expresion": {
                        "type": "string",
                        "description": "Expresión matemática a evaluar (ej: 2+2, sqrt(16))"
                    }
                },
                "required": ["expresion"]
            }
        ),
        types.Tool(
            name="es_primo",
            description="Verifica si un número es primo",
            inputSchema={
                "type": "object",
                "properties": {
                    "numero": {"type": "integer", "description": "Número a verificar"}
                },
                "required": ["numero"]
            }
        )
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    """Ejecuta la herramienta solicitada."""
    if name == "calcular":
        try:
            # Evaluar expresión de forma segura
            allowed_names = {k: v for k, v in math.__dict__.items() 
                           if not k.startswith(\'_\')}
            result = eval(arguments["expresion"], {"__builtins__": {}}, allowed_names)
            return [types.TextContent(type="text", text=str(result))]
        except Exception as e:
            return [types.TextContent(type="text", text=f"Error: {e}")]
    
    elif name == "es_primo":
        n = arguments["numero"]
        if n < 2:
            resultado = False
        else:
            resultado = all(n % i != 0 for i in range(2, int(n**0.5) + 1))
        return [types.TextContent(type="text", 
                                   text=f"{n} {'ES' if resultado else \'NO ES\'} primo")]

@app.list_resources()
async def list_resources() -> list[types.Resource]:
    """Lista recursos disponibles."""
    return [
        types.Resource(
            uri="math://constants",
            name="Constantes Matemáticas",
            description="Constantes matemáticas importantes",
            mimeType="application/json"
        )
    ]

@app.read_resource()
async def read_resource(uri: str) -> str:
    """Lee un recurso por URI."""
    if str(uri) == "math://constants":
        constants = {
            "pi": math.pi,
            "e": math.e,
            "tau": math.tau,
            "inf": float(\'inf\')
        }
        return json.dumps(constants, indent=2)
    raise ValueError(f"Recurso no encontrado: {uri}")

@app.list_prompts()
async def list_prompts() -> list[types.Prompt]:
    """Lista prompts disponibles."""
    return [
        types.Prompt(
            name="resolver_problema",
            description="Template para resolver problemas matemáticos paso a paso",
            arguments=[
                types.PromptArgument(
                    name="problema",
                    description="El problema matemático a resolver",
                    required=True
                )
            ]
        )
    ]

@app.get_prompt()
async def get_prompt(name: str, arguments: dict) -> types.GetPromptResult:
    """Retorna un prompt por nombre."""
    if name == "resolver_problema":
        problema = arguments.get("problema", "")
        return types.GetPromptResult(
            description="Resolver problema paso a paso",
            messages=[
                types.PromptMessage(
                    role="user",
                    content=types.TextContent(
                        type="text",
                        text=f"""Resuelve este problema matemático paso a paso:

Problema: {problema}

Instrucciones:
1. Identifica los datos del problema
2. Determina las operaciones necesarias
3. Resuelve paso a paso
4. Verifica el resultado
5. Presenta la respuesta final claramente"""
                    )
                )
            ]
        )

async def main():
    async with stdio_server() as (read_stream, write_stream):
        await app.run(
            read_stream,
            write_stream,
            app.create_initialization_options()
        )

if __name__ == "__main__":
    import asyncio
    asyncio.run(main())
'''

with open('../07_model_context_protocol/mcp_math_server.py', 'w') as f:
    f.write(mcp_server_code)

print("✅ Servidor MCP guardado en: 07_model_context_protocol/mcp_math_server.py")

## 2. Cliente MCP

In [ ]:
# Cliente MCP que se conecta al servidor y usa Claude para procesar
mcp_client_code = '''
# mcp_client.py
# Cliente MCP que usa Claude para interactuar con el servidor

import asyncio
import json
import os
from dotenv import load_dotenv
import anthropic
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
claude = anthropic.Anthropic()

async def run_mcp_client():
    # Conectar al servidor MCP
    server_params = StdioServerParameters(
        command="python",
        args=["mcp_math_server.py"],
    )
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 1. Listar herramientas disponibles
            tools_result = await session.list_tools()
            print("Herramientas disponibles:")
            for tool in tools_result.tools:
                print(f"  - {tool.name}: {tool.description}")
            
            # 2. Convertir herramientas MCP a formato Anthropic
            anthropic_tools = [
                {
                    "name": tool.name,
                    "description": tool.description,
                    "input_schema": tool.inputSchema
                }
                for tool in tools_result.tools
            ]
            
            # 3. Bucle de conversación con herramientas
            messages = [{
                "role": "user",
                "content": "¿Cuánto es sqrt(256) + 37? Y también dime si 97 es primo."
            }]
            
            while True:
                response = claude.messages.create(
                    model="claude-sonnet-4-5",
                    max_tokens=1024,
                    tools=anthropic_tools,
                    messages=messages
                )
                
                if response.stop_reason == "end_turn":
                    print("\nClaude:", response.content[0].text)
                    break
                
                if response.stop_reason == "tool_use":
                    messages.append({"role": "assistant", "content": response.content})
                    tool_results = []
                    
                    for block in response.content:
                        if block.type == "tool_use":
                            print(f"Usando: {block.name}({block.input})")
                            # Llamar herramienta en el servidor MCP
                            result = await session.call_tool(block.name, block.input)
                            print(f"Resultado: {result.content[0].text}")
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": result.content[0].text
                            })
                    
                    messages.append({"role": "user", "content": tool_results})

if __name__ == "__main__":
    asyncio.run(run_mcp_client())
'''

with open('../07_model_context_protocol/mcp_client.py', 'w') as f:
    f.write(mcp_client_code)

print("✅ Cliente MCP guardado en: 07_model_context_protocol/mcp_client.py")
print("\nPara ejecutar:")
print("  cd 07_model_context_protocol")
print("  python mcp_client.py")

## 3. Resumen de MCP

| Componente | Descripción | Método |
|------------|-------------|--------|
| **Tools** | Funciones ejecutables | `@app.list_tools()`, `@app.call_tool()` |
| **Resources** | Datos legibles | `@app.list_resources()`, `@app.read_resource()` |
| **Prompts** | Templates de prompts | `@app.list_prompts()`, `@app.get_prompt()` |

### Casos de uso
- Conectar Claude con bases de datos empresariales
- Integrar con APIs externas de forma estandarizada
- Compartir herramientas entre múltiples clientes
- Crear ecosistemas de herramientas reutilizables

## 4. MCP con Claude Desktop

Para usar el servidor MCP con Claude Desktop, añade en `~/Library/Application Support/Claude/claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "math-server": {
      "command": "python",
      "args": ["/ruta/absoluta/mcp_math_server.py"]
    }
  }
}
```